In [1]:
# ============================================================
# GRID t-SNE 2D + GMM OTIMIZADO
# CSV RESUMO: UMA LINHA POR PERPLEXIDADE
# SAÍDA FIXA: 2x2_tsne_score.csv
# ============================================================

import time
import inspect
import warnings
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    log_loss
)

warnings.filterwarnings("ignore")


# ============================================================
# BARRA DE PROGRESSO
# ============================================================

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def detectar_target(df):
    if "status_fraude" in df.columns:
        return "status_fraude"

    if "Class" in df.columns:
        return "Class"

    raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")


def criar_tsne_compat(
    perplexity,
    random_state=42,
    init="pca",
    max_iter=300,
    learning_rate="auto",
    n_jobs=-1,
    verbose=0,
    method="barnes_hut",
    angle=0.5
):
    """
    Cria TSNE compatível com versões diferentes do sklearn.
    Algumas versões usam max_iter; outras usam n_iter.
    """

    assinatura = inspect.signature(TSNE)
    parametros = assinatura.parameters

    kwargs = {
        "n_components": 2,
        "perplexity": perplexity,
        "random_state": random_state,
        "init": init,
        "learning_rate": learning_rate,
        "method": method,
        "angle": angle,
        "verbose": verbose
    }

    if "max_iter" in parametros:
        kwargs["max_iter"] = max_iter
    else:
        kwargs["n_iter"] = max_iter

    if "n_jobs" in parametros:
        kwargs["n_jobs"] = n_jobs

    return TSNE(**kwargs)


def formatar_tempo(segundos):
    segundos = int(segundos)

    h = segundos // 3600
    m = (segundos % 3600) // 60
    s = segundos % 60

    if h > 0:
        return f"{h}h {m}min {s}s"

    if m > 0:
        return f"{m}min {s}s"

    return f"{s}s"


def calcular_mcc_vetorizado(tp, fp, fn, tn):
    """
    Calcula MCC de forma vetorizada.
    """

    numerador = (tp * tn) - (fp * fn)

    denominador = np.sqrt(
        (tp + fp) *
        (tp + fn) *
        (tn + fp) *
        (tn + fn)
    )

    mcc = np.divide(
        numerador,
        denominador,
        out=np.zeros_like(numerador, dtype=float),
        where=denominador != 0
    )

    return mcc


def encontrar_melhor_corte_mcc(y_true, prob_fraude):
    """
    Encontra o melhor ponto de corte maximizando MCC.
    Versão vetorizada e rápida.
    """

    y_true = np.asarray(y_true).astype(int)
    prob_fraude = np.asarray(prob_fraude, dtype=float)

    precision, recall, thresholds = precision_recall_curve(
        y_true,
        prob_fraude
    )

    if len(thresholds) == 0:
        return 0.5, 0.0

    precision = precision[:-1]
    recall = recall[:-1]

    total_pos = np.sum(y_true == 1)
    total_neg = np.sum(y_true == 0)

    tp = recall * total_pos

    fp = np.zeros_like(tp, dtype=float)

    mask_precision = precision > 0

    fp[mask_precision] = tp[mask_precision] * (
        (1.0 / precision[mask_precision]) - 1.0
    )

    fn = total_pos - tp
    tn = total_neg - fp

    mccs = calcular_mcc_vetorizado(
        tp=tp,
        fp=fp,
        fn=fn,
        tn=tn
    )

    melhor_idx = int(np.nanargmax(mccs))

    melhor_corte = thresholds[melhor_idx]
    melhor_mcc = mccs[melhor_idx]

    return float(melhor_corte), float(melhor_mcc)


def calcular_log_veross_gmm(gmm, X):
    """
    Log-verossimilhança total da GMM.
    """

    log_prob = gmm.score_samples(X)

    return float(np.sum(log_prob))


def calcular_neg_log_veross_com_rotulo(prob_fraude, y_true):
    """
    Negative log-likelihood usando o rótulo real.

    Se y = 1, usa P(fraude).
    Se y = 0, usa 1 - P(fraude).
    """

    eps = 1e-15

    prob_fraude = np.clip(
        prob_fraude,
        eps,
        1 - eps
    )

    y_true = np.asarray(y_true).astype(int)

    log_veross = (
        y_true * np.log(prob_fraude)
        + (1 - y_true) * np.log(1 - prob_fraude)
    )

    return float(-np.sum(log_veross))


def calcular_log_loss_norm(log_loss_valor):
    """
    Normalização simples do log loss.
    Quanto menor o log loss, mais próximo de 1.
    """

    return float(1 / (1 + log_loss_valor))


def gerar_score_final(auc_pr, mcc, log_loss_norm):
    """
    Score final usando apenas:
    - AUC-PR
    - MCC
    - Log Loss normalizado
    """

    score = (
        0.45 * auc_pr
        + 0.40 * max(mcc, 0)
        + 0.15 * log_loss_norm
    )

    return float(score)


# ============================================================
# FUNÇÃO PRINCIPAL DO GRID
# ============================================================

def gerar_grid_tsne_2d_scores_resumo(
    arquivo_dados="creditcard.csv",
    arquivo_saida="2x2_tsne_score.csv",
    pasta_saida=".",
    perplexidades=(30, 50),
    random_state=42,
    init="pca",
    max_iter=300,
    learning_rate="auto",
    n_jobs_tsne=-1,
    features_tsne=None,
    target_name=None,
    gmm_n_components=2,
    gmm_n_init=3,
    gmm_random_state=42,
    gmm_covariance_type="full",
    reg_covar=1e-6,
    escalar_antes_tsne=True,
    escalar_tsne_para_gmm=True,
    verbose_tsne=0,
    method_tsne="barnes_hut",
    angle_tsne=0.5,
    salvar_parcial=True,
    arquivo_parcial="2x2_tsne_score_parcial.csv"
):

    tempo_inicio_total = time.time()

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_saida = pasta_saida / arquivo_saida
    caminho_parcial = pasta_saida / arquivo_parcial

    # ========================================================
    # LEITURA
    # ========================================================

    df = pd.read_csv(arquivo_dados)

    if target_name is None:
        target_name = detectar_target(df)

    if target_name not in df.columns:
        raise ValueError(f"Target '{target_name}' não encontrado no arquivo.")

    # ========================================================
    # FEATURES
    # ========================================================

    if features_tsne is None:
        features_tsne = [
            col for col in df.columns
            if col != target_name and pd.api.types.is_numeric_dtype(df[col])
        ]

    if len(features_tsne) == 0:
        raise ValueError("Nenhuma feature numérica encontrada para o t-SNE.")

    dados = df[
        features_tsne + [target_name]
    ].dropna().copy()

    X_original = dados[features_tsne].copy()
    y = dados[target_name].astype(int).to_numpy()

    valores_target = sorted(np.unique(y))

    if set(valores_target) != {0, 1}:
        raise ValueError(
            f"O target precisa ser binário com valores 0 e 1. "
            f"Valores encontrados: {valores_target}"
        )

    n_amostras = X_original.shape[0]

    perplexidades = list(perplexidades)

    perplexidades_validas = [
        p for p in perplexidades
        if p < n_amostras
    ]

    if len(perplexidades_validas) == 0:
        raise ValueError(
            "Nenhuma perplexidade válida. "
            "A perplexidade precisa ser menor que o número de amostras."
        )

    scaler_antes_tsne_nome = "StandardScaler" if escalar_antes_tsne else "None"
    scaler_tsne_para_gmm_nome = "StandardScaler" if escalar_tsne_para_gmm else "None"

    print("=" * 80)
    print("GRID t-SNE 2D + GMM - CSV RESUMO COM REPLICAÇÃO")
    print("=" * 80)
    print(f"Arquivo: {arquivo_dados}")
    print(f"CSV final: {arquivo_saida}")
    print(f"CSV parcial: {arquivo_parcial}")
    print(f"Target: {target_name}")
    print(f"Amostras usadas: {n_amostras:,}".replace(",", "."))
    print(f"Features usadas no t-SNE: {len(features_tsne)}")
    print(f"Perplexidades válidas: {perplexidades_validas}")
    print(f"max_iter: {max_iter}")
    print(f"init: {init}")
    print(f"learning_rate: {learning_rate}")
    print(f"random_state t-SNE: {random_state}")
    print(f"n_jobs_tsne: {n_jobs_tsne}")
    print(f"method_tsne: {method_tsne}")
    print(f"angle_tsne: {angle_tsne}")
    print(f"scaler antes t-SNE: {scaler_antes_tsne_nome}")
    print(f"scaler t-SNE para GMM: {scaler_tsne_para_gmm_nome}")
    print(f"GMM random_state: {gmm_random_state}")
    print("=" * 80)

    # ========================================================
    # ESCALONAMENTO ANTES DO t-SNE
    # ========================================================

    if escalar_antes_tsne:
        scaler_features = StandardScaler()
        X_tsne_input = scaler_features.fit_transform(X_original)
    else:
        X_tsne_input = X_original.to_numpy()

    resultados = []

    barra = tqdm(
        perplexidades_validas,
        desc="Grid t-SNE 2D",
        unit="perplexidade",
        leave=True
    )

    # ========================================================
    # LOOP DAS PERPLEXIDADES
    # ========================================================

    for idx, perplexity in enumerate(barra, start=1):

        tempo_inicio = time.time()

        barra.set_postfix(
            {
                "perplexity": perplexity,
                "status": "t-SNE"
            }
        )

        print()
        print("-" * 80)
        print(f"[{idx}/{len(perplexidades_validas)}] Rodando t-SNE | perplexity={perplexity}")
        print("-" * 80)

        # ====================================================
        # t-SNE
        # ====================================================

        tsne = criar_tsne_compat(
            perplexity=perplexity,
            random_state=random_state,
            init=init,
            max_iter=max_iter,
            learning_rate=learning_rate,
            n_jobs=n_jobs_tsne,
            verbose=verbose_tsne,
            method=method_tsne,
            angle=angle_tsne
        )

        X_tsne = tsne.fit_transform(X_tsne_input)

        n_iter_real = getattr(tsne, "n_iter_", np.nan)

        barra.set_postfix(
            {
                "perplexity": perplexity,
                "status": "GMM"
            }
        )

        # ====================================================
        # ESCALONAMENTO DO t-SNE PARA GMM
        # ====================================================

        if escalar_tsne_para_gmm:
            scaler_gmm = StandardScaler()
            X_gmm = scaler_gmm.fit_transform(X_tsne)
        else:
            X_gmm = X_tsne

        # ====================================================
        # GMM
        # ====================================================

        gmm = GaussianMixture(
            n_components=gmm_n_components,
            covariance_type=gmm_covariance_type,
            random_state=gmm_random_state,
            n_init=gmm_n_init,
            reg_covar=reg_covar
        )

        gmm.fit(X_gmm)

        clusters = gmm.predict(X_gmm)
        responsabilidades = gmm.predict_proba(X_gmm)

        tabela_cluster = pd.crosstab(
            clusters,
            y
        )

        if 1 not in tabela_cluster.columns:
            raise ValueError("A classe fraude, valor 1, não foi encontrada no target.")

        cluster_fraude = int(tabela_cluster[1].idxmax())

        prob_fraude = responsabilidades[:, cluster_fraude]

        prob_fraude = np.clip(
            prob_fraude,
            1e-15,
            1 - 1e-15
        )

        # ====================================================
        # MÉTRICAS
        # ====================================================

        auc_pr = average_precision_score(
            y,
            prob_fraude
        )

        log_loss_valor = log_loss(
            y,
            prob_fraude,
            labels=[0, 1]
        )

        log_loss_norm = calcular_log_loss_norm(
            log_loss_valor
        )

        melhor_ponto_corte, mcc = encontrar_melhor_corte_mcc(
            y_true=y,
            prob_fraude=prob_fraude
        )

        ponto_corte_medio = 0.5

        log_veross_gmm = calcular_log_veross_gmm(
            gmm=gmm,
            X=X_gmm
        )

        neg_log_veross_gmm = -log_veross_gmm

        neg_log_veross_com_rotulo = calcular_neg_log_veross_com_rotulo(
            prob_fraude=prob_fraude,
            y_true=y
        )

        diferenca_neg_log_veross = (
            neg_log_veross_com_rotulo
            - neg_log_veross_gmm
        )

        score_final = gerar_score_final(
            auc_pr=auc_pr,
            mcc=mcc,
            log_loss_norm=log_loss_norm
        )

        tempo_execucao = time.time() - tempo_inicio
        tempo_formatado = formatar_tempo(tempo_execucao)

        # ====================================================
        # RESULTADO
        # ====================================================

        resultado = {
            "Perplexity": int(perplexity),
            "N_Components_TSNE": 2,
            "Max_Iter": int(max_iter),
            "N_Iter_Real": n_iter_real,
            "Init": init,
            "Learning_Rate": learning_rate,
            "Random_State_TSNE": random_state,
            "Method_TSNE": method_tsne,
            "Angle_TSNE": float(angle_tsne),
            "N_Jobs_TSNE": n_jobs_tsne,
            "Scaler_Antes_TSNE": scaler_antes_tsne_nome,
            "Scaler_TSNE_Para_GMM": scaler_tsne_para_gmm_nome,

            "GMM_N_Components": int(gmm_n_components),
            "GMM_Covariance_Type": gmm_covariance_type,
            "GMM_N_Init": int(gmm_n_init),
            "GMM_Random_State": gmm_random_state,
            "GMM_Reg_Covar": float(reg_covar),

            "AUC_PR": float(auc_pr),
            "MCC": float(mcc),
            "Log_Loss": float(log_loss_valor),
            "Log_Loss_Norm": float(log_loss_norm),

            "Neg_Log_Veross_Com_Rotulo": float(neg_log_veross_com_rotulo),
            "Neg_Log_Veross_GMM": float(neg_log_veross_gmm),
            "Diferenca_Neg_Log_Veross": float(diferenca_neg_log_veross),

            "Score_Final": float(score_final),
            "Melhor_Ponto_Corte": float(melhor_ponto_corte),
            "Ponto_Corte_Medio": float(ponto_corte_medio),

            "Tempo": tempo_formatado
        }

        resultados.append(resultado)

        # ====================================================
        # SALVAR PARCIAL
        # ====================================================

        df_parcial = pd.DataFrame(resultados)

        df_parcial = df_parcial.sort_values(
            by="Score_Final",
            ascending=False
        ).reset_index(drop=True)

        if salvar_parcial:
            df_parcial.to_csv(
                caminho_parcial,
                index=False
            )

        barra.set_postfix(
            {
                "perplexity": perplexity,
                "score": f"{score_final:.4f}",
                "tempo": tempo_formatado
            }
        )

        print(f"Perplexity: {perplexity}")
        print(f"AUC-PR: {auc_pr:.6f}")
        print(f"MCC: {mcc:.6f}")
        print(f"Log Loss: {log_loss_valor:.6f}")
        print(f"Log Loss Norm: {log_loss_norm:.6f}")
        print(f"Score Final: {score_final:.6f}")
        print(f"Melhor ponto de corte: {melhor_ponto_corte:.6f}")
        print(f"Tempo: {tempo_formatado}")

        if salvar_parcial:
            print(f"Parcial salvo em: {caminho_parcial.resolve()}")

    # ========================================================
    # CSV FINAL
    # ========================================================

    df_resultados = pd.DataFrame(resultados)

    df_resultados = df_resultados.sort_values(
        by="Score_Final",
        ascending=False
    ).reset_index(drop=True)

    colunas_ordenadas = [
        "Perplexity",
        "N_Components_TSNE",
        "Max_Iter",
        "N_Iter_Real",
        "Init",
        "Learning_Rate",
        "Random_State_TSNE",
        "Method_TSNE",
        "Angle_TSNE",
        "N_Jobs_TSNE",
        "Scaler_Antes_TSNE",
        "Scaler_TSNE_Para_GMM",

        "GMM_N_Components",
        "GMM_Covariance_Type",
        "GMM_N_Init",
        "GMM_Random_State",
        "GMM_Reg_Covar",

        "AUC_PR",
        "MCC",
        "Log_Loss",
        "Log_Loss_Norm",

        "Neg_Log_Veross_Com_Rotulo",
        "Neg_Log_Veross_GMM",
        "Diferenca_Neg_Log_Veross",

        "Score_Final",
        "Melhor_Ponto_Corte",
        "Ponto_Corte_Medio",

        "Tempo"
    ]

    df_resultados = df_resultados[colunas_ordenadas]

    df_resultados.to_csv(
        caminho_saida,
        index=False
    )

    tempo_total = time.time() - tempo_inicio_total

    print()
    print("=" * 80)
    print("GRID FINALIZADO")
    print("=" * 80)
    print(f"CSV resumo salvo em: {caminho_saida.resolve()}")
    print(f"Linhas no CSV final: {len(df_resultados)}")
    print(f"Colunas no CSV final: {len(df_resultados.columns)}")
    print(f"Tempo total: {formatar_tempo(tempo_total)}")
    print("=" * 80)

    return df_resultados


# ============================================================
# FUNÇÃO PERSONALIZADA
# ============================================================

def rodar_grid_tsne_2d_personalizado(
    perplexity_inicio=10,
    perplexity_fim=100,
    perplexity_passo=10,
    arquivo_dados="creditcard.csv",
    pasta_saida=".",
    arquivo_saida="2x2_tsne_score.csv",
    random_state=42,
    init="pca",
    max_iter=300,
    learning_rate="auto",
    n_jobs_tsne=-1,
    target_name=None,
    verbose_tsne=0,
    method_tsne="barnes_hut",
    angle_tsne=0.5,
    gmm_n_components=2,
    gmm_n_init=3,
    gmm_random_state=42,
    gmm_covariance_type="full",
    reg_covar=1e-6,
    escalar_antes_tsne=True,
    escalar_tsne_para_gmm=True
):

    if perplexity_passo <= 0:
        raise ValueError("perplexity_passo precisa ser maior que zero.")

    if perplexity_fim < perplexity_inicio:
        raise ValueError("perplexity_fim precisa ser maior ou igual a perplexity_inicio.")

    perplexidades = list(
        range(
            perplexity_inicio,
            perplexity_fim + 1,
            perplexity_passo
        )
    )

    arquivo_parcial = arquivo_saida.replace(
        ".csv",
        "_parcial.csv"
    )

    print("=" * 80)
    print("GRID t-SNE 2D PERSONALIZADO")
    print("=" * 80)
    print(f"Perplexidades geradas: {perplexidades}")
    print(f"Arquivo final: {arquivo_saida}")
    print(f"Arquivo parcial: {arquivo_parcial}")
    print("=" * 80)

    return gerar_grid_tsne_2d_scores_resumo(
        arquivo_dados=arquivo_dados,
        arquivo_saida=arquivo_saida,
        pasta_saida=pasta_saida,
        perplexidades=perplexidades,
        random_state=random_state,
        init=init,
        max_iter=max_iter,
        learning_rate=learning_rate,
        n_jobs_tsne=n_jobs_tsne,
        target_name=target_name,
        verbose_tsne=verbose_tsne,
        method_tsne=method_tsne,
        angle_tsne=angle_tsne,
        gmm_n_components=gmm_n_components,
        gmm_n_init=gmm_n_init,
        gmm_random_state=gmm_random_state,
        gmm_covariance_type=gmm_covariance_type,
        reg_covar=reg_covar,
        escalar_antes_tsne=escalar_antes_tsne,
        escalar_tsne_para_gmm=escalar_tsne_para_gmm,
        salvar_parcial=True,
        arquivo_parcial=arquivo_parcial
    )

In [2]:
df_grid = rodar_grid_tsne_2d_personalizado(
    perplexity_inicio=10,
    perplexity_fim=100,
    perplexity_passo=10,
    arquivo_dados="creditcard.csv",
    pasta_saida=".",
    arquivo_saida="2x2_tsne_score.csv",
    max_iter=300,
    n_jobs_tsne=-1,
    init="pca",
    learning_rate="auto",
    random_state=42,
    gmm_random_state=42,
    target_name=None,
    verbose_tsne=0
)

df_grid.head()

GRID t-SNE 2D PERSONALIZADO
Perplexidades geradas: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
Arquivo final: 2x2_tsne_score.csv
Arquivo parcial: 2x2_tsne_score_parcial.csv
GRID t-SNE 2D + GMM - CSV RESUMO COM REPLICAÇÃO
Arquivo: creditcard.csv
CSV final: 2x2_tsne_score.csv
CSV parcial: 2x2_tsne_score_parcial.csv
Target: status_fraude
Amostras usadas: 283.726
Features usadas no t-SNE: 30
Perplexidades válidas: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
max_iter: 300
init: pca
learning_rate: auto
random_state t-SNE: 42
n_jobs_tsne: -1
method_tsne: barnes_hut
angle_tsne: 0.5
scaler antes t-SNE: StandardScaler
scaler t-SNE para GMM: StandardScaler
GMM random_state: 42


Grid t-SNE 2D:   0%|          | 0/10 [00:00<?, ?perplexidade/s]


--------------------------------------------------------------------------------
[1/10] Rodando t-SNE | perplexity=10
--------------------------------------------------------------------------------
Perplexity: 10
AUC-PR: 0.015537
MCC: 0.157528
Log Loss: 1.683133
Log Loss Norm: 0.372699
Score Final: 0.125907
Melhor ponto de corte: 0.998403
Tempo: 53min 46s
Parcial salvo em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\2x2_tsne_score_parcial.csv

--------------------------------------------------------------------------------
[2/10] Rodando t-SNE | perplexity=20
--------------------------------------------------------------------------------
Perplexity: 20
AUC-PR: 0.006880
MCC: 0.086969
Log Loss: 1.634066
Log Loss Norm: 0.379641
Score Final: 0.094830
Melhor ponto de corte: 0.999282
Tempo: 51min 50s
Parcial salvo em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE

MemoryError: Unable to allocate 654. MiB for an array with shape (85685252,) and data type float64